# 01. Exploratory Data Analysis (EDA)
## DNS Exfiltration and Tunneling Detection

This notebook explores benign and malicious DNS query patterns to uncover distinguishing statistical characteristics, including query lengths, character distributions, and Shannon entropy.


In [ ]:
import os
import sys
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append("..")
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")


In [ ]:
# Load configuration
with open("../configs/config.yaml", "r") as f:
    config = yaml.safe_load(f)

data_path = "../" + config["paths"]["features_dataset_csv"]
print(f"Target dataset: {data_path}")


In [ ]:
# Load data if available, or generate synthetic sample for demonstration
if os.path.exists(data_path):
    df = pd.read_csv(data_path)
else:
    print("Processed dataset not found yet. Using synthetic sample for EDA demonstration...")
    np.random.seed(42)
    n_samples = 1000
    df = pd.DataFrame({
        "domain_len": np.concatenate([np.random.normal(18, 5, n_samples // 2), np.random.normal(55, 10, n_samples // 2)]),
        "entropy": np.concatenate([np.random.normal(2.8, 0.4, n_samples // 2), np.random.normal(4.2, 0.3, n_samples // 2)]),
        "subdomain_count": np.concatenate([np.random.poisson(1, n_samples // 2), np.random.poisson(4, n_samples // 2)]),
        "digit_ratio": np.concatenate([np.random.beta(1, 20, n_samples // 2), np.random.beta(5, 10, n_samples // 2)]),
        "label": [0] * (n_samples // 2) + [1] * (n_samples // 2)
    })
df.head()


In [ ]:
# Class distribution
print("Class distribution:")
print(df["label"].value_counts(normalize=True))


In [ ]:
# Visualizing Distributions (Query Length & Entropy)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(data=df, x="domain_len", hue="label", kde=True, ax=axes[0], palette="Set2")
axes[0].set_title("Domain Length Distribution (0: Benign, 1: Exfil)")

sns.histplot(data=df, x="entropy", hue="label", kde=True, ax=axes[1], palette="Set2")
axes[1].set_title("Shannon Entropy Distribution (0: Benign, 1: Exfil)")
plt.tight_layout()
plt.show()


In [ ]:
# Correlation Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Feature Correlation Matrix")
plt.show()
